# Deep Learning Lab 0
In this lab we will be training Convolutional Neural Networks and Visualize the progress through Wandb.
# Task 0.1

## Imports

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
from torch.utils.data import random_split

import torchvision.models as models
from torchvision.models import alexnet, AlexNet_Weights

import wandb
import os

# empty cache
torch.cuda.empty_cache()


## Wandb Setup


In [ ]:
# Storage of wandb legend data
xs = []
ys_train = []
ys_valid = []



## Preprocessing
### Checklist Train Data
- Normalization
- Data Augmentation
- Create images --> ToTensor()
- Dataloader
### Checklist Test Data
- Normalization
- Create images --> ToTensor()

In [ ]:
"""
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4), # Randomly Crops 32x32 Region After Padding To Create Small Shifts --> Robustness
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), 
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
    ])

"""

transformer = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
    ])

transformer_MNIST = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize(mean=(0.5), std=(0.5))
    ])

transformer_SVHN = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5), std=(0.5))
])

traindata = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True, 
    download=True, 
    transform=transformer)

testdata = torchvision.datasets.CIFAR10(
    root='./data', 
    train=False, 
    download=True, 
    transform=transformer)

In [ ]:
train_size = int(0.8*len(traindata))
val_size = len(traindata) - train_size

train_dataset, val_dataset = random_split(traindata, [train_size, val_size], generator=torch.Generator().manual_seed(42))

trainload = torch.utils.data.DataLoader(traindata, batch_size=32, shuffle=True)
valload = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)
testload = torch.utils.data.DataLoader(testdata, batch_size=32, shuffle=False)

## Default settings

In [ ]:
def_epochs = 500
def_patience = 20 # innan 20

## Training and Testing 
### Checklist
- Activation Function LeakyReLU
- Optimizer Stochastic Gradient Descent lr = 0.0001
- Accuracy on Test set
- Val/Train Stopping when the Val loss converges
- Tensorboard and wandb
- Schedulers


In [ ]:
# setting up device for training
if torch.cuda.is_available():
    device = "cuda" #Nvidia Graphics Card
elif torch.backends.mps.is_available():
    device = "mps" # Apple
else:
    device = "cpu" # Worst Case
print(device)


In [13]:
class CNN_LReLU(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN_LReLU, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Images: 32x32
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2), # Images: (32x32)/(2*2) --> 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2) # Images  (16x16)/(2x2) --> 8x8
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128), # 4096 --> 128
            nn.LeakyReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

class CNN_MNIST(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN_MNIST, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), # Images: 32x32
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2), # Images: (32x32)/(2*2) --> 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2) # Images  (16x16)/(2x2) --> 8x8
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 128), # 4096 --> 128
            nn.LeakyReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

### Settings CNN SGD Leaky ReLU

In [ ]:
model_sgd_LReLU = CNN_LReLU(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model_sgd_LReLU.parameters(), lr=0.001)

epochs = def_epochs
ran_epochs = []
early_stop_patience = def_patience
early_stopping_counter = 0

In [ ]:
# ------------------------- Wandb -------------------------
wandb.init(project="DeepLearning", name="CIFAR10 SGD LeakyReLU")

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode="min",
        factor = 0.1, 
        patience = 3 # Wait x amount of epochs before reducing lr
    )

best_val_loss = float('inf')

for epoch in range(epochs):
    ran_epochs.append(epoch+1)
    # ------------------------- TRAIN -------------------------
    model_sgd_LReLU.train()
    running_loss = 0.0

    for images, labels in trainload:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model_sgd_LReLU(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        

    average_loss = running_loss / len(trainload)
    ys_train.append(average_loss)

    # ------------------------- VALIDATION -------------------------

    model_sgd_LReLU.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model_sgd_LReLU(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

            

        average_val_loss = val_running_loss / len(valload)
        ys_valid.append(average_val_loss)
        
    scheduler.step(average_val_loss)

    # ------------------------- Saving Best Model and EarlyStopping -------------------------
    if best_val_loss*0.99 > average_val_loss: 
        best_val_loss = average_val_loss
        early_stopping_counter = 0
        torch.save(model_sgd_LReLU.state_dict(), "BestModel_SGDLeakyReLU.pth")
        print("Saved The Best Performing Model")
    else:
        early_stopping_counter += 1
        print(f"No improvement in loss, increasing counter:{early_stopping_counter}")

    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.8f}"
    )

    if early_stopping_counter > early_stop_patience:
        print("Early stop initiated")
        break

wandb.log({
    "Loss - CIFAR10_SGD_LeakyReLU": wandb.plot.line_series(
        xs=ran_epochs,
        ys=[ys_train, ys_valid],
        keys=["Training Loss", "Validation Loss"]
    )#,
    #"epochs": xs,
    #"train loss": ys_train,
    #"val loss": ys_valid
}) 

wandb.finish()
ys_train = []
ys_valid = []


In [ ]:
model_sgd_LReLU.load_state_dict(torch.load("BestModel.pth"))
model_sgd_LReLU.to(device)
model_sgd_LReLU.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model_sgd_LReLU(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()        

average_test_loss = test_loss / len(testload)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")



### Checklist 2
- LeakyReLU --> Tanh
- Optimizer: SGD --> Adam
- Visualize the results on a Tensorboard

In [ ]:
class CNN_tanh(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN_tanh, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Images: 32x32
            nn.BatchNorm2d(32),
            nn.Tanh(),
            nn.MaxPool2d(2,2), # Images: (32x32)/(2*2) --> 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.Tanh(),
            nn.MaxPool2d(2,2) # Images  (16x16)/(2x2) --> 8x8
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128), # 4096 --> 128
            nn.Tanh(),
            nn.Dropout(0.1),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

### Settings Adam SGD Leaky ReLU

In [ ]:
# the first model with LeakyReLU activation function and Adam optimizer
model_adam_LReLU = CNN_LReLU(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_adam_LReLU.parameters(), lr=0.001)

epochs = def_epochs
ran_epochs = []
early_stop_patience = def_patience
early_stopping_counter = 0

In [18]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode="min",
        factor = 0.1, 
        patience = 3 # Wait x amount of epochs before reducing lr
    )

In [ ]:
wandb.init(project="DeepLearning", name="CIFAR10 Adam LeakyReLU")


# Initialize loss tracking lists
ys_train = []
ys_valid = []

# ------------------------- Wandb -------------------------



best_val_loss = float('inf') 

for epoch in range(epochs):
    ran_epochs.append(epoch+1)
    # ------------------------- TRAIN -------------------------
    model_adam_LReLU.train()
    running_loss = 0.0

    for images, labels in trainload:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model_adam_LReLU(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        

    average_loss = running_loss / len(trainload)
    ys_train.append(average_loss)
    
    # ------------------------- VALIDATION -------------------------

    model_adam_LReLU.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model_adam_LReLU(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

        average_val_loss = val_running_loss / len(valload)
        ys_valid.append(average_val_loss)
        
    scheduler.step(average_loss)
    
    # ------------------------- Saving Best Model and EarlyStopping -------------------------
    if best_val_loss*0.99 > average_val_loss: 
        best_val_loss = average_val_loss
        early_stopping_counter = 0
        torch.save(model_adam_LReLU.state_dict(), "BestModel_Adam_LeakyReLU.pth")
        print("Saved The Best Performing Model")
    else:
        early_stopping_counter += 1
        print(f"No improvement in loss, increasing counter:{early_stopping_counter}")

    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )

    # Early Stopping
    if early_stopping_counter > early_stop_patience:
        print("Early stop initiated")
        break

# Log combined plot at end of training
wandb.log({
    "Loss - CIFAR10 Learning with Adam + ReLU": wandb.plot.line_series(
        xs=ran_epochs,
        ys=[ys_train, ys_valid],
        keys=["Training Loss", "Validation Loss"],
    ),
})
wandb.finish()
ys_train = []
ys_valid = []


In [ ]:
model_sgd_LReLU.load_state_dict(torch.load("BestModel_Adam_LeakyReLU.pth"))
model_sgd_LReLU.to(device)
model_sgd_LReLU.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model_sgd_LReLU(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()         

average_test_loss = test_loss / len(testload)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")



### Settings Adam SGD tanh

In [ ]:
# the first (CNN) model with tanh activation function and Adam optimizer
model_tanh = CNN_tanh(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_tanh.parameters(), lr=0.001) 

epochs = def_epochs
ran_epochs = []
early_stop_patience = def_patience
early_stopping_counter = 0

In [ ]:
# ------------------------- Wandb -------------------------
wandb.init(project="DeepLearning", name="CIFAR10 Adam tanh")



best_val_loss = float('inf')
    

for epoch in range(epochs):
    ran_epochs.append(epoch+1)
    # ------------------------- TRAIN -------------------------
    model_tanh.train()
    running_loss = 0.0

    for images, labels in trainload:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model_tanh(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        

    average_loss = running_loss / len(trainload)
    ys_train.append(average_loss)
    
    # ------------------------- VALIDATION -------------------------

    model_tanh.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model_tanh(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

        average_val_loss = val_running_loss / len(valload)
        ys_valid.append(average_val_loss)
        
    scheduler.step(average_loss)

    # ------------------------- Saving Best Model and EarlyStopping-------------------------
    if best_val_loss*0.99 > average_val_loss: 
        best_val_loss = average_val_loss
        early_stopping_counter = 0
        torch.save(model_tanh.state_dict(), "BestModel_Adam_Tanh.pth")
        print("Saved The Best Performing Model")
    else:
        early_stopping_counter += 1
        print(f"No improvement in loss, increasing counter:{early_stopping_counter}")

    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )

    # Early Stopping
    if early_stopping_counter > early_stop_patience:
        print("Early stop initiated")
        break

# Log combined plot at end of training
wandb.log({
    "Loss - CIFAR10 Learning with Adam + Tanh": wandb.plot.line_series(
        xs=ran_epochs,
        ys=[ys_train, ys_valid],
        keys=["Training Loss", "Validation Loss"],
    ),
})
wandb.finish()

ys_train = []
ys_valid = []


In [ ]:
model_tanh.load_state_dict(torch.load("BestModel_Task_AdamTanh.pth"))
model_tanh.to(device)
model_tanh.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model_tanh(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()         

average_test_loss = test_loss / len(testload)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")



# Task 0.2

## 0.2 Task
### 0.2.1 Transfer Learning from ImageNet
- Download and prepare CIFAR-10 dataset (it is already available in the above mentioned libraries)
- Use AlexNet as the model (Pytorch AlexNet)
- You have to perform two separate experiments-
    - Train the model for CIFAR-10 data, Report the test test accuracy. (also referred as fine tuning the model)
    - Use the pretarined weights of AlexNet, in other words use AlexNet as a pretrained network for image classification on CIFAR-10 data (also referred as Feature Extraction), Report the test test accuracy. (optional)
- In both the above cases remember to add an extra fully connected layer to the classifier with number of neurons = 10, because there are 10 classes in CIFAR-10 dataset. This layer will be trainable in both cases.
- Explain (briefly!) what is the difference between the two runs and why there is a difference in performance. (optional)

In [19]:
early_stop_patience = def_patience
early_stopping_counter = 0
epochs = def_epochs
ran_epochs = []
learning_rate = 0.001

In [20]:
print(device)
print(torch.cuda.is_available())

wandb.init(project="DeepLearning", name="CIFAR10 FineTuning AlexNet")

# Initialize loss tracking lists
ys_train = []
ys_valid = []

train_transformer = transforms.Compose([
    transforms.Resize((256)),
    transforms.RandomCrop((227)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2023, 0.1994, 0.2010))
    ])  

val_test_transformer = transforms.Compose([
    transforms.Resize((256)),
    transforms.CenterCrop((227)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2023, 0.1994, 0.2010))
    ])

full_trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True) 
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=val_test_transformer)

# Download CIFAR-10 datasets
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transformer)
valset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=val_test_transformer)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=val_test_transformer)


train_size = int(0.8 * len(full_trainset)) 
val_size = len(full_trainset) - train_size

train_indices, val_indices = torch.utils.data.random_split(range(len(full_trainset)), [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_dataset = torch.utils.data.Subset(torchvision.datasets.CIFAR10(root='./data', train=True, transform=train_transformer), train_indices.indices) 
val_dataset = torch.utils.data.Subset(torchvision.datasets.CIFAR10(root='./data', train=True, transform=val_test_transformer), val_indices.indices)


trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
valloader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"Dataset Split - Train: {train_size}, Val: {val_size}, Test: {len(testset)}")


print("\n" + "=" * 50)
print("EXPERIMENT 1: Fine-Tuning Pretrained AlexNet")
print("=" * 50)

model_finetune = models.alexnet(weights=AlexNet_Weights.IMAGENET1K_V1)
# Replace the last fully connected layer to output 10 classes (CIFAR-10)
model_finetune.classifier[6] = nn.Linear(4096, 10)
model_finetune = model_finetune.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_finetune = torch.optim.Adam(model_finetune.parameters(), lr=learning_rate)

# Training loop for fine-tuning
best_val_loss = float('inf')

for epoch in range(epochs):
    ran_epochs.append(epoch+1)
    model_finetune.train()
    running_loss = 0.0
    
    for images, labels in trainloader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer_finetune.zero_grad()
        
        outputs = model_finetune(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer_finetune.step()
        
        running_loss += loss.item()
    
    avg_train_loss = running_loss / len(trainloader)
    ys_train.append(avg_train_loss)

    # Validation
    model_finetune.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in valloader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model_finetune(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
    
    average_val_loss = val_loss / len(valloader)
    ys_valid.append(average_val_loss)
    
    loss_gap = average_val_loss - avg_train_loss
    print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f} | Val Loss: {average_val_loss:.4f} | Loss Gap: {loss_gap:.4f}")
    
    # Save best model
    if best_val_loss > average_val_loss: 
        best_val_loss = average_val_loss
        early_stopping_counter = 0
        torch.save(model_finetune.state_dict(), "BestModel_CIFAR10_Finetune.pth")
        print("Saved The Best Performing Model")
    else:
        early_stopping_counter += 1
        print(f"No improvement in loss, increasing counter:{early_stopping_counter}")

    if early_stopping_counter > early_stop_patience:
        print("Early stop initiated")
        break

# Test the fine-tuned model
model_finetune.load_state_dict(torch.load("BestModel_Finetune.pth"))
model_finetune.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model_finetune(images)
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_finetune = 100 * (correct / total)
print(f"\nTest Accuracy (Fine-tuning): {accuracy_finetune:.2f}%")

# Log combined plot at end of training
wandb.log({
    "Loss - AlexNet FineTuneing": wandb.plot.line_series(
        xs=ran_epochs,
        ys=[ys_train, ys_valid],
        keys=["Training Loss", "Validation Loss"],
    )
})
wandb.finish()

ys_train = []
ys_valid = []


cuda
True


/usr/local/lib/python3.11/dist-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Dataset Split - Train: 40000, Val: 10000, Test: 10000

EXPERIMENT 1: Fine-Tuning Pretrained AlexNet
Epoch [1/500] Train Loss: 1.7667 | Val Loss: 1.4128 | Loss Gap: -0.3540
Saved The Best Performing Model
Epoch [2/500] Train Loss: 1.3436 | Val Loss: 1.1518 | Loss Gap: -0.1918
Saved The Best Performing Model
Epoch [3/500] Train Loss: 1.1706 | Val Loss: 1.0626 | Loss Gap: -0.1080
Saved The Best Performing Model
Epoch [4/500] Train Loss: 1.0920 | Val Loss: 0.9745 | Loss Gap: -0.1174
Saved The Best Performing Model
Epoch [5/500] Train Loss: 1.0246 | Val Loss: 0.9098 | Loss Gap: -0.1148
Saved The Best Performing Model
Epoch [6/500] Train Loss: 0.9810 | Val Loss: 0.8334 | Loss Gap: -0.1475
Saved The Best Performing Model
Epoch [7/500] Train Loss: 0.9564 | Val Loss: 0.8470 | Loss Gap: -0.1094
No improvement in loss, increasing counter:1
Epoch [8/500] Train Loss: 0.9333 | Val Loss: 0.8440 | Loss Gap: -0.0893
No improvement in loss, increasing counter:2
Epoch [9/500] Train Loss: 0.9135 | Val Los

In [21]:
early_stop_patience = def_patience
early_stopping_counter = 0
epochs = def_epochs
ran_epochs = []
learning_rate = 0.001

In [22]:
# ======================== EXPERIMENT 2: Feature Extraction (Frozen Pretrained AlexNet) ========================
wandb.init(project="DeepLearning", name="CIFAR10 Feature Extraction AlexNet")

print("\n" + "=" * 50)
print("EXPERIMENT 2: Feature Extraction (Frozen AlexNet)")
print("=" * 50)

model_feature_extract = models.alexnet(pretrained=True)

# Freeze all layers except the final classifier layer
for param in model_feature_extract.features.parameters():
    param.requires_grad = False

# Only the newly added layer will be trainable
model_feature_extract.classifier[6] = nn.Linear(4096, 10)
model_feature_extract = model_feature_extract.to(device)

# Only optimize the new classifier layer
optimizer_feature_extract = torch.optim.Adam(model_feature_extract.classifier[6].parameters(), lr=learning_rate)

# Training loop for feature extraction
best_val_loss = float('inf')

for epoch in range(epochs):
    ran_epochs.append(epoch+1)
    
    model_feature_extract.train()
    running_loss = 0.0
    
    for images, labels in trainloader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer_feature_extract.zero_grad()
        
        outputs = model_feature_extract(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer_feature_extract.step()
        
        running_loss += loss.item()
    
    avg_train_loss = running_loss / len(trainloader)
    ys_train.append(avg_train_loss)

    # Validation
    model_feature_extract.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in valloader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model_feature_extract(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(valloader)
    
    ys_valid.append(avg_val_loss)
    
    loss_gap = avg_val_loss - avg_train_loss
    print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Loss Gap: {loss_gap:.4f}")
    
    # Save best model
    if best_val_loss > average_val_loss: 
        best_val_loss = average_val_loss
        early_stopping_counter = 0
        torch.save(model_feature_extract.state_dict(), "BestModel_CIFAR10_FeatureExtract.pth")
        print("Saved The Best Performing Model")
    else:
        early_stopping_counter += 1
        print(f"No improvement in loss, increasing counter:{early_stopping_counter}")

    if early_stopping_counter > early_stop_patience:
        print("Early stop initiated")
        break

# Test the feature extraction model
model_feature_extract.load_state_dict(torch.load("BestModel_FeatureExtract.pth"))
model_feature_extract.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model_feature_extract(images)
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_feature_extract = 100 * (correct / total)
print(f"\nTest Accuracy (Feature Extraction): {accuracy_feature_extract:.2f}%")
wandb.log({"feature_extract_test_accuracy": accuracy_feature_extract})

# ======================== COMPARISON ========================
print("\n" + "=" * 50)
print("COMPARISON SUMMARY")
print("=" * 50)
print(f"Fine-Tuning Test Accuracy:      {accuracy_finetune:.2f}%")
print(f"Feature Extraction Accuracy:    {accuracy_feature_extract:.2f}%")
print(f"Difference:                     {abs(accuracy_finetune - accuracy_feature_extract):.2f}%")
print("\nExplanation:")
print("- Fine-tuning allows all layers to be updated with CIFAR-10 data,")
print("  adapting the learned ImageNet features to the new task.")
print("- Feature extraction keeps the pre-trained weights frozen,")
print("  using ImageNet features as fixed representations.")
print("- Fine-tuning typically performs better as it can specialize")
print("  the model filters to the specific CIFAR-10 task.")

# Log combined plot
wandb.log({
    "Loss - AlexNet Feature Extraction": wandb.plot.line_series(
        xs=ran_epochs,
        ys=[ys_train, ys_valid],
        keys=["Training Loss", "Validation Loss"],
    )
})
wandb.finish()
ys_train = []
ys_valid = []



EXPERIMENT 2: Feature Extraction (Frozen AlexNet)


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch [1/500] Train Loss: 0.8165 | Val Loss: 0.6403 | Loss Gap: -0.1762
Saved The Best Performing Model
Epoch [2/500] Train Loss: 0.7220 | Val Loss: 0.5894 | Loss Gap: -0.1326
No improvement in loss, increasing counter:1
Epoch [3/500] Train Loss: 0.6918 | Val Loss: 0.5757 | Loss Gap: -0.1162
No improvement in loss, increasing counter:2
Epoch [4/500] Train Loss: 0.6829 | Val Loss: 0.5774 | Loss Gap: -0.1055
No improvement in loss, increasing counter:3
Epoch [5/500] Train Loss: 0.6795 | Val Loss: 0.5866 | Loss Gap: -0.0929
No improvement in loss, increasing counter:4
Epoch [6/500] Train Loss: 0.6790 | Val Loss: 0.5790 | Loss Gap: -0.1000
No improvement in loss, increasing counter:5
Epoch [7/500] Train Loss: 0.6800 | Val Loss: 0.5856 | Loss Gap: -0.0943
No improvement in loss, increasing counter:6
Epoch [8/500] Train Loss: 0.6701 | Val Loss: 0.5666 | Loss Gap: -0.1035
No improvement in loss, increasing counter:7
Epoch [9/500] Train Loss: 0.6669 | Val Loss: 0.5511 | Loss Gap: -0.1158
No im

feature_extract_test_accuracy,▁
feature_extract_test_accuracy,66.83


### Task 0.2.2

Prepare a CNN of your choice and train it on the MNIST data. Report the accuracy

In [23]:
#Task 2.2
traindata_MNIST = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transformer_MNIST)

testdata_MNIST = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transformer_MNIST)


train_size = int(0.8*len(traindata_MNIST))
val_size = len(traindata_MNIST) - train_size

train_dataset_MNIST, val_dataset_MNIST = random_split(traindata_MNIST, [train_size, val_size], generator=torch.Generator().manual_seed(42))

trainload_MNIST = torch.utils.data.DataLoader(traindata_MNIST, batch_size=32, shuffle=True)
valload_MNIST = torch.utils.data.DataLoader(val_dataset_MNIST, batch_size=32, shuffle=False)
testload_MNIST = torch.utils.data.DataLoader(testdata_MNIST, batch_size=32, shuffle=False)

model = CNN_MNIST(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)

In [24]:
early_stop_patience = def_patience
early_stopping_counter = 0
epochs = def_epochs
ran_epochs = []

In [25]:
wandb.init(project="DeepLearning", name="MNIST Training with CNN")

best_val_loss = float('inf')

for epoch in range(epochs):
    ran_epochs.append(epoch+1)
    # ------------------------- TRAIN -------------------------
    model.train()
    running_loss = 0.0

    for images, labels in trainload_MNIST:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(trainload_MNIST)
    ys_train.append(average_loss)
    
    # ------------------------- VALIDATION -------------------------

    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload_MNIST:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

        average_val_loss = val_running_loss / len(valload_MNIST)
        ys_valid.append(average_val_loss)

    scheduler.step(average_val_loss)

    # ------------------------- Saving Best Model and EarlyStop -------------------------
    if best_val_loss*0.99 > average_val_loss: 
        best_val_loss = average_val_loss
        early_stopping_counter = 0
        torch.save(model.state_dict(), "BestModel_MNIST.pth")
        print("Saved The Best Performing Model")
    else:
        early_stopping_counter += 1
        print(f"No improvement in loss, increasing counter:{early_stopping_counter}")

    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )

    if early_stopping_counter > early_stop_patience:
        print("Early stop initiated")
        break
    
# Log combined plot
wandb.log({
    "Loss - MNIST Learning with CNN": wandb.plot.line_series(
        xs=list(range(epochs)),
        ys=[ys_train, ys_valid],
        keys=["Training Loss", "Validation Loss"]
    )
})
wandb.finish()
ys_train = []
ys_valid = []


Saved The Best Performing Model
Epoch [1/500] train loss: 1.898, val loss: 1.489, lr: 0.000100
Saved The Best Performing Model
Epoch [2/500] train loss: 1.191, val loss: 0.932, lr: 0.000100
Saved The Best Performing Model
Epoch [3/500] train loss: 0.798, val loss: 0.658, lr: 0.000100
Saved The Best Performing Model
Epoch [4/500] train loss: 0.599, val loss: 0.516, lr: 0.000100
Saved The Best Performing Model
Epoch [5/500] train loss: 0.484, val loss: 0.422, lr: 0.000100
Saved The Best Performing Model
Epoch [6/500] train loss: 0.411, val loss: 0.364, lr: 0.000100
Saved The Best Performing Model
Epoch [7/500] train loss: 0.361, val loss: 0.321, lr: 0.000100
Saved The Best Performing Model
Epoch [8/500] train loss: 0.323, val loss: 0.290, lr: 0.000100
Saved The Best Performing Model
Epoch [9/500] train loss: 0.294, val loss: 0.265, lr: 0.000100
Saved The Best Performing Model
Epoch [10/500] train loss: 0.272, val loss: 0.245, lr: 0.000100
Saved The Best Performing Model
Epoch [11/500] tr

In [26]:
#Test best model for MNIST
model.load_state_dict(torch.load("BestModel_MNIST.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload_MNIST:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()        

average_test_loss = test_loss / len(testload_MNIST)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")


Test loss: 0.029
Test accuracy: 99.05%


Use the above model as a pre-trained CNN for the SVHN dataset. Report the accuracy

In [27]:
#Import SVHN Dataset 0.2.2
traindata_SVHN = torchvision.datasets.SVHN(
    root='./data',
    split="train",
    download=True,
    transform=transformer_SVHN)

testdata_SVHN = torchvision.datasets.SVHN(
    root='./data',
    split="test",
    download=True,
    transform=transformer_SVHN)


train_size = int(0.8*len(traindata_SVHN))
val_size = len(traindata_SVHN) - train_size

train_dataset_SVHN, val_dataset_SVHN = random_split(traindata_SVHN, [train_size, val_size], generator=torch.Generator().manual_seed(42))

trainload_SVHN = torch.utils.data.DataLoader(traindata_SVHN, batch_size=32, shuffle=True)
valload_SVHN = torch.utils.data.DataLoader(val_dataset_SVHN, batch_size=32, shuffle=False)
testload_SVHN = torch.utils.data.DataLoader(testdata_SVHN, batch_size=32, shuffle=False)

In [28]:
#Test MNIST model on SVHN dataset

#Test best model for MNIST
model.load_state_dict(torch.load("BestModel_MNIST.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload_SVHN:
        images = images.to(device)
        if images.shape[1] == 3:  # RGB
            images = images.mean(dim=1, keepdim=True)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

average_test_loss = test_loss / len(testload_SVHN)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")




Test loss: 2.619
Test accuracy: 25.39%


In the third step you are performing transfer learning from MNIST to SVHN.

In [41]:
early_stop_patience = def_patience
early_stopping_counter = 0
epochs = def_epochs
ran_epochs = []

lr = 0.005 # från 0.0001 -> 0.01
ys_lr = []

In [42]:
#Transfer learning from MNIST to SVHN with feature extraction
wandb.init(project="DeepLearning", name="MNIST Transfer learning to SVHN - Feature extraction")

model.load_state_dict(torch.load("BestModel_MNIST.pth"))
model.to(device)

#Freeze model weights
for param in model.parameters():
    param.requires_grad = False

model.classifier[4] = nn.Linear(128, 10)
model = model.to(device)
optimizer = torch.optim.SGD(model.classifier[4].parameters(), lr=lr) 

best_val_loss = float('inf')

for epoch in range(epochs):
    ran_epochs.append(epoch+1)
    ys_lr.append(optimizer.param_groups[0]['lr'])

    # ------------------------- TRAIN -------------------------
    model.train()
    running_loss = 0.0

    for images, labels in trainload_SVHN:
        images = images.to(device)
        if images.shape[1] == 3:  # RGB
            images = images.mean(dim=1, keepdim=True)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(trainload_SVHN)
    ys_train.append(average_loss)

    # ------------------------- VALIDATION -------------------------

    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload_SVHN:
            images = images.to(device)
            if images.shape[1] == 3:  # RGB
                images = images.mean(dim=1, keepdim=True)
            labels = labels.to(device)

            outputs = model(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

        average_val_loss = val_running_loss / len(valload_SVHN)
        ys_valid.append(average_val_loss)

    scheduler.step(average_val_loss)

    # ------------------------- Saving Best Model and EarlyStopping -------------------------
    if best_val_loss > average_val_loss: 
        best_val_loss = average_val_loss
        early_stopping_counter = 0
        torch.save(model.state_dict(), "BestModel_SGDReLu.pth")
        print("Saved The Best Performing Model")
    else:
        early_stopping_counter += 1
        print(f"No improvement in loss, increasing counter:{early_stopping_counter}")

    if early_stopping_counter > early_stop_patience:
        print("Early stop initiated")
        break

    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )

wandb.log({
    "Loss: MNIST Transfer learning to SVHN - Feature extraction": wandb.plot.line_series(
        xs=list(range(epochs)),
        ys=[ys_train, ys_valid, ys_lr],
        keys=["Training Loss", "Validation Loss", "Learning Rate"]
    )
})
wandb.finish()

ys_train = []
ys_valid = []
ys_lr = []

Saved The Best Performing Model
Epoch [1/500] train loss: 1.765, val loss: 1.560, lr: 0.005000
Saved The Best Performing Model
Epoch [2/500] train loss: 1.558, val loss: 1.458, lr: 0.005000
Saved The Best Performing Model
Epoch [3/500] train loss: 1.509, val loss: 1.428, lr: 0.005000
Saved The Best Performing Model
Epoch [4/500] train loss: 1.483, val loss: 1.385, lr: 0.005000
Saved The Best Performing Model
Epoch [5/500] train loss: 1.464, val loss: 1.372, lr: 0.005000
Saved The Best Performing Model
Epoch [6/500] train loss: 1.452, val loss: 1.350, lr: 0.005000
Saved The Best Performing Model
Epoch [7/500] train loss: 1.450, val loss: 1.349, lr: 0.005000
Saved The Best Performing Model
Epoch [8/500] train loss: 1.438, val loss: 1.337, lr: 0.005000
Saved The Best Performing Model
Epoch [9/500] train loss: 1.432, val loss: 1.331, lr: 0.005000
No improvement in loss, increasing counter:1
Epoch [10/500] train loss: 1.428, val loss: 1.341, lr: 0.005000
No improvement in loss, increasing c

In [43]:
#Test transfer model on SVHN dataset with feature extraction

#Test best model for MNIST
model.load_state_dict(torch.load("BestModel_SVHN_Feature_Extraction.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload_SVHN:
        images = images.to(device)
        if images.shape[1] == 3:  # RGB
            images = images.mean(dim=1, keepdim=True)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

average_test_loss = test_loss / len(testload_SVHN)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")



Test loss: 2.258
Test accuracy: 19.13%


In [44]:
early_stop_patience = def_patience
early_stopping_counter = 0
epochs = def_epochs
ran_epochs = []

lr = 0.001 # från 0.0001
ys_lr = []

In [45]:
# Now transfer learning with fine-tuning:
wandb.init(project="DeepLearning", name="MNIST Transfer learning to SVHN - Fine tuning")

model.load_state_dict(torch.load("BestModel_MNIST.pth"))
model.to(device)


for param in model.parameters():
    param.requires_grad = True

model.classifier[4] = nn.Linear(128, 10)
model = model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

best_val_loss = float('inf')

for epoch in range(epochs):
    ran_epochs.append(epoch+1)
    ys_lr.append(optimizer.param_groups[0]['lr'])

    # ------------------------- TRAIN -------------------------
    model.train()
    running_loss = 0.0

    for images, labels in trainload_SVHN:
        images = images.to(device)
        if images.shape[1] == 3:  # RGB
            images = images.mean(dim=1, keepdim=True)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(trainload_SVHN)
    ys_train.append(average_loss)

    # ------------------------- VALIDATION -------------------------

    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload_SVHN:
            images = images.to(device)
            if images.shape[1] == 3:  # RGB
                images = images.mean(dim=1, keepdim=True)
            labels = labels.to(device)

            outputs = model(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

        average_val_loss = val_running_loss / len(valload_SVHN)
        ys_valid.append(average_val_loss)
        
    scheduler.step(average_val_loss)

    # ------------------------- Saving Best Model and EarlyStopping -------------------------
    if best_val_loss*0.99 > average_val_loss: 
        best_val_loss = average_val_loss
        early_stopping_counter = 0
        torch.save(model.state_dict(), "BestModel_SVHN_Transfer.pth")
        print("Saved The Best Performing Model")
    else:
        early_stopping_counter += 1
        print(f"No improvement in loss, increasing counter:{early_stopping_counter}")

    if early_stopping_counter > early_stop_patience:
        print("Early stop initiated")
        break
        
    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )

    
wandb.log({
    "Loss: MNIST Transfer learning to SVHN - Fine tuning": wandb.plot.line_series(
        xs=list(range(epochs)),
        ys=[ys_train, ys_valid, ys_lr],
        keys=["Training Loss", "Validation Loss", "Learning Rate"]
    )
})
wandb.finish()


Saved The Best Performing Model
Epoch [1/500] train loss: 1.345, val loss: 0.853, lr: 0.001000
Saved The Best Performing Model
Epoch [2/500] train loss: 0.768, val loss: 0.666, lr: 0.001000
Saved The Best Performing Model
Epoch [3/500] train loss: 0.655, val loss: 0.597, lr: 0.001000
Saved The Best Performing Model
Epoch [4/500] train loss: 0.597, val loss: 0.555, lr: 0.001000
Saved The Best Performing Model
Epoch [5/500] train loss: 0.559, val loss: 0.523, lr: 0.001000
No improvement in loss, increasing counter:1
Epoch [6/500] train loss: 0.532, val loss: 0.526, lr: 0.001000
No improvement in loss, increasing counter:2
Epoch [7/500] train loss: 0.508, val loss: 0.529, lr: 0.001000
Saved The Best Performing Model
Epoch [8/500] train loss: 0.489, val loss: 0.460, lr: 0.001000
Saved The Best Performing Model
Epoch [9/500] train loss: 0.470, val loss: 0.438, lr: 0.001000
No improvement in loss, increasing counter:1
Epoch [10/500] train loss: 0.456, val loss: 0.444, lr: 0.001000
Saved The 

In [46]:
#Test transfer model on SVHN dataset with fine tuning

#Test best model for MNIST
model.load_state_dict(torch.load("BestModel_SVHN_Fine_Tuning.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload_SVHN:
        images = images.to(device)
        if images.shape[1] == 3:  # RGB
            images = images.mean(dim=1, keepdim=True)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

average_test_loss = test_loss / len(testload_SVHN)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")



Test loss: 2.094
Test accuracy: 28.47%
